# Feature Engineering & Diagnóstico Avanzado

- El análisis exploratorio inicial `(01_EDA)` confirmó que el dataset tiene desbalance severo, faltantes y una dependencia lineal débil entre la mayoría de las variables y el churn. 

Sin embargo, para construir un modelo predictivo robusto y alineado con la lógica de negocio, es obligatorio ir más allá de las correlaciones simples y aplicar un enfoque de `Feature Engineering` y `Diagnóstico Estructural`.

### Justificación profesional de esta extensión:

**1. Recencia y Antigüedad (Temporalidad):** 

- El comportamiento de compra no es estático. 
- La Recencia (días desde la última compra) es el predictor más fuerte de abandono en retail/ e-commerce, y la Antigüedad (días desde el registro) captura la lealtad acumulada. Ignorar las fechas es desaprovechar la señal más valiosa.

**2. Análisis de Categóricas (Chi-cuadrado):** 

- La correlación de Spearman no es válida para variables nominales (país, canal). 
- Necesitamos saber si, por ejemplo, los clientes de Alemania o los que usan suscripción anual tienen una tasa de churn significativamente diferente. 

Sin este paso, estamos ciegos a segmentos completos.

**3. Multicolinealidad (VIF):** 

- Incluir lifetime_value junto con total_spent es redundante (están casi perfectamente correlacionados). 

- Esto infla la varianza de los coeficientes y enrarece la interpretación de modelos lineales. 

Debemos detectar y eliminar la redundancia.

**4. Ingeniería de Interacciones:** 

- El churn no depende de una sola variable, sino de la relación entre ellas. 

- Un cliente que gasta poco pero tiene muchas visitas es muy diferente de uno que gasta poco y tiene pocas visitas. 

Creamos variables derivadas para capturar estas dinámicas.

**5. Segmentación de Riesgo (Lógica de Negocio):** 

- El 84.7% de clientes son "No Churn", pero dentro de ellos hay subgrupos: los "Leales" (compran seguido y gastan mucho) y los "Desconectados" (no compran hace meses). 

Crear una variable de "riesgo" basada en reglas de negocio (Recencia + Gasto) permite al modelo aprender patrones de abandono incluso dentro de la mayoría silenciosa.

# Celda 1: — Importación de librerías y configuración global


In [1]:
# =====================================================
# 02_Preprocessing.ipynb
# Preparación de datos, Feature Engineering y Diagnóstico
# Dataset: Sales and Marketing DataSet (Kaggle)
# =====================================================

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Configuración de visualización (opcional, para seguimiento)
sns.set_theme(style="whitegrid", context="notebook")

# Rutas
PROCESSED_DATA_PATH = "../data/processed/data_eda_clean.csv"
ENRICHED_DATA_PATH = "../data/processed/data_enriched.csv"

print("✅ Librerías y rutas configuradas para el notebook 02")

✅ Librerías y rutas configuradas para el notebook 02


# Celda 2 — Carga del dataset limpio desde el punto de control

In [2]:
# =====================================================
# Celda 2: Carga del dataset limpio (punto de control)
# =====================================================
print("📂 Cargando dataset desde el punto de control...")

eda = pd.read_csv(PROCESSED_DATA_PATH)
print(f"✅ Dataset cargado correctamente. Dimensiones: {eda.shape}")

# Verificar que las columnas clave existen para evitar errores posteriores
print("\n📋 Primeras columnas del dataset:")
print(eda.columns.tolist()[:10])  # Muestra las primeras 10 para confirmar

📂 Cargando dataset desde el punto de control...
✅ Dataset cargado correctamente. Dimensiones: (15000, 30)

📋 Primeras columnas del dataset:
['customer_id', 'gender', 'age', 'country', 'city', 'signup_date', 'last_purchase_date', 'acquisition_channel', 'device_type', 'subscription_type']


## Bloque 1: Transformación de Fechas y Variables Temporales

- Las fechas están en formato object. 

- Las convertimos a datetime para poder calcular diferencias. 

La recencia es el tiempo que ha pasado desde la última compra; si no hay compra reciente, la probabilidad de churn se dispara. 

La antigüedad mide cuánto tiempo lleva el cliente en la empresa.

In [3]:
# =====================================================
# Bloque 1: Transformación de Fechas y Variables Temporales
# =====================================================
print("📅 Transformando fechas y creando variables temporales...")

# Convertir a datetime
eda['signup_date'] = pd.to_datetime(eda['signup_date'])
eda['last_purchase_date'] = pd.to_datetime(eda['last_purchase_date'])

# Fecha de referencia (la última fecha de compra en el dataset)
ref_date = eda['last_purchase_date'].max()
print(f"Fecha de referencia (última compra registrada): {ref_date.date()}")

# Recencia (días sin comprar)
eda['recency_days'] = (ref_date - eda['last_purchase_date']).dt.days

# Antigüedad (días desde el registro)
eda['tenure_days'] = (ref_date - eda['signup_date']).dt.days

# Verificar resultados
print(f"Recencia - Min: {eda['recency_days'].min()}, Max: {eda['recency_days'].max()}, Media: {eda['recency_days'].mean():.1f}")
print(f"Antigüedad - Min: {eda['tenure_days'].min()}, Max: {eda['tenure_days'].max()}, Media: {eda['tenure_days'].mean():.1f}")

📅 Transformando fechas y creando variables temporales...
Fecha de referencia (última compra registrada): 2025-03-10
Recencia - Min: 0, Max: 799, Media: 399.3
Antigüedad - Min: 165, Max: 1164, Media: 664.1


### Interpretación Variables temporales

- Recencia: rango 0–799 días, media 399.3 días. 

- La mitad de los clientes llevan más de un año sin comprar (lo cual es alarmante).

- Antigüedad: rango 165–1164 días, media 664 días (~1.8 años).

**Implicación:** 
La recencia tiene una variabilidad enorme y captura el abandono silencioso. Esta variable será clave en el modelo.

---

# Bloque 2: Análisis de Asociación de Categóricas (Chi-cuadrado)

- Verificamos si las variables categóricas (country, subscription_type, acquisition_channel, etc.) tienen una relación estadísticamente significativa con churn. 

Si el p-valor es < 0.05, la variable es dependiente y debe mantenerse. Esto es más riguroso que confiar solo en la intuición.

In [4]:
# =====================================================
# Bloque 2: Análisis de Asociación (Chi-cuadrado vs Churn)
# =====================================================
from scipy.stats import chi2_contingency

print("\n📊 Evaluando dependencia de variables categóricas con Churn...")

categorical_vars = ['gender', 'country', 'city', 'subscription_type', 'device_type', 'acquisition_channel', 'payment_method']
chi2_results = []

for var in categorical_vars:
    if var in eda.columns:
        crosstab = pd.crosstab(eda[var], eda['churn'])
        chi2, p, dof, expected = chi2_contingency(crosstab)
        chi2_results.append({
            'Variable': var,
            'Chi-cuadrado': chi2,
            'p-valor': p,
            'Significativo': 'Sí' if p < 0.05 else 'No'
        })

chi2_df = pd.DataFrame(chi2_results).sort_values('p-valor')
display(chi2_df)

print("\n✅ Variables con p-valor < 0.05 son significativas y se mantienen.")


📊 Evaluando dependencia de variables categóricas con Churn...


,Variable,Chi-cuadrado,p-valor,Significativo
0,gender,8.150207,0.016990,Sí
1,country,6.093699,0.192259,No
4,device_type,1.709092,0.425476,No
3,subscription_type,0.531017,0.466179,No
6,payment_method,2.841812,0.584637,No
2,city,3.994598,0.677407,No
5,acquisition_channel,2.134272,0.711079,No



✅ Variables con p-valor < 0.05 son significativas y se mantienen.


## Interpretación Chi-cuadrado (categóricas vs churn)

- Solo gender resultó significativo (p=0.017).

- country, city, subscription_type, device_type, acquisition_channel y payment_method no mostraron asociación estadística con el churn (p > 0.05).


#### **Conclusión:** 

- En el pipeline de preprocesamiento, se puede eliminar todas las categóricas excepto gender, o mantenerlas y dejar que el modelo decida su relevancia. 
- Dado que el VIF no aplica a categóricas, mantenerlas no perjudica, pero sabiendo que no son significativas, podrían eliminarse para simplificar el modelo. 
- Se recomienda mantenerlas y que la selección de características (SelectKBest) las descarte si no aportan.

---

# Bloque 3: Evaluación de Multicolinealidad (VIF)

- El Factor de Inflación de la Varianza (VIF) mide cuánto se infla la varianza de un coeficiente de regresión debido a la correlación con otras variables. 

Un VIF > 5 (o 10) indica alta multicolinealidad. Por ejemplo, `lifetime_value` y `total_spent` probablemente miden lo mismo. Si dos variables tienen VIF alto, eliminamos una para reducir ruido.

In [5]:
# =====================================================
# Bloque 3: Evaluación de Multicolinealidad (VIF)
# =====================================================
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("\n📈 Calculando VIF para variables numéricas...")

# Seleccionar numéricas (excluyendo ID y target)
num_vars_for_vif = ['age', 'total_visits', 'avg_session_time', 'pages_per_session',
                    'email_open_rate', 'email_click_rate', 'total_spent', 'avg_order_value',
                    'discount_used', 'support_tickets', 'refund_requested', 'delivery_delay_days',
                    'satisfaction_score', 'nps_score', 'marketing_spend_per_user', 'lifetime_value',
                    'last_3_month_purchase_freq', 'recency_days', 'tenure_days']

X_vif = eda[num_vars_for_vif].dropna()
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_data = vif_data.sort_values("VIF", ascending=False)
display(vif_data)

# Identificar variables con VIF > 10
high_vif = vif_data[vif_data["VIF"] > 10]
if not high_vif.empty:
    print(f"\n⚠️ Variables con VIF > 10 (alta multicolinealidad):")
    display(high_vif)
    print("📌 Se recomienda eliminar una de cada par redundante (ej. lifetime_value vs total_spent).")
else:
    print("\n✅ No se detectaron variables con VIF > 10. El dataset es estructuralmente saludable.")


📈 Calculando VIF para variables numéricas...


,Variable,VIF
1,total_visits,13.445790
0,age,11.471241
12,satisfaction_score,10.372381
3,pages_per_session,7.705809
2,avg_session_time,7.612780
14,marketing_spend_per_user,6.502054
7,avg_order_value,6.435197
18,tenure_days,5.909594
15,lifetime_value,4.354314
5,email_click_rate,3.903560



⚠️ Variables con VIF > 10 (alta multicolinealidad):


,Variable,VIF
1,total_visits,13.445790
0,age,11.471241
12,satisfaction_score,10.372381


📌 Se recomienda eliminar una de cada par redundante (ej. lifetime_value vs total_spent).


## Interpretación de Multicolinealidad (VIF)

- total_visits (13.45), age (11.47) y satisfaction_score (10.37) superan el umbral de 10.

**Esto indica que estas variables están altamente correlacionadas con otras (ej. age con tenure_days; total_visits con avg_session_time).**

#### Acción: 

- Eliminar total_visits y age del conjunto de predictores, ya que su información está contenida en otras variables (como recency_days o tenure_days). 
- satisfaction_score se mantiene por su alta relevancia predictiva (correlación -0.26 con churn), a pesar del VIF elevado, porque es un predictor directo.

---


# Bloque 4: Ingeniería de Características (Interacciones)

- Creamos variables que capturen relaciones entre dos o más variables base. 

Estas suelen tener mayor poder predictivo que las variables aisladas porque reflejan comportamientos compuestos (eficiencia, densidad de quejas, etc.).

In [6]:
# =====================================================
# Bloque 4: Ingeniería de Características (Interacciones)
# =====================================================
print("\n⚙️ Creando variables de interacción...")

# Gasto promedio por visita (eficiencia de conversión)
eda['avg_spend_per_visit'] = eda['total_spent'] / (eda['total_visits'] + 1e-6)

# Tickets por visita (frustración relativa)
eda['ticket_rate'] = eda['support_tickets'] / (eda['total_visits'] + 1e-6)

# Ratio de recencia vs antigüedad (qué % de la vida del cliente lleva sin comprar)
eda['recency_tenure_ratio'] = eda['recency_days'] / (eda['tenure_days'] + 1)

# Gasto por ticket de soporte (clientes que gastan mucho pero abren muchos tickets son "quejosos valiosos")
eda['spend_per_ticket'] = eda['total_spent'] / (eda['support_tickets'] + 1)

print("✅ Variables creadas: avg_spend_per_visit, ticket_rate, recency_tenure_ratio, spend_per_ticket")


⚙️ Creando variables de interacción...
✅ Variables creadas: avg_spend_per_visit, ticket_rate, recency_tenure_ratio, spend_per_ticket


## Interpretación de Variables de interacción

- Se crearon avg_spend_per_visit, ticket_rate, recency_tenure_ratio y spend_per_ticket.


#### Implicación: 

- Estas variables capturan eficiencia, frustración relativa y proporción de vida sin comprar. 
- Deberían tener mejor poder predictivo que las variables originales aisladas. Se mantienen todas.4

---

# Bloque 5: Segmentación de Riesgo (Lógica de Negocio)

- El 84.7% de los clientes son No Churn, pero no todos son iguales. 

- Basándonos en la lógica de negocio (Recencia + Gasto), clasificamos a los clientes en niveles de riesgo. 

Esto crea una variable categórica que el modelo puede usar para identificar patrones de abandono incipiente incluso antes de que ocurra el churn oficial.

In [7]:
# =====================================================
# Bloque 5: Segmentación de Riesgo (Lógica de Negocio)
# =====================================================
print("\n🎯 Creando segmento de riesgo basado en Recencia y Gasto...")

# Calcular percentiles o umbrales (ej. mediana)
median_spent = eda['total_spent'].median()
median_recency = eda['recency_days'].median()

# Reglas de negocio
def assign_risk(row):
    if row['recency_days'] <= 30 and row['total_spent'] >= median_spent:
        return 'Leal'  # Bajo riesgo
    elif row['recency_days'] <= 90 and row['total_spent'] >= median_spent * 0.5:
        return 'Regular'  # Riesgo medio
    elif row['recency_days'] > 90 and row['total_spent'] < median_spent:
        return 'Desconectado'  # Alto riesgo
    else:
        return 'Mixto'  # Casos intermedios

eda['risk_segment'] = eda.apply(assign_risk, axis=1)

# Ver distribución
print("\nDistribución de segmentos de riesgo:")
display(eda['risk_segment'].value_counts())

# Validar lógica: ver si el segmento se relaciona con churn
print("\nTasa de churn por segmento de riesgo:")
display(eda.groupby('risk_segment')['churn'].mean().sort_values(ascending=False))


🎯 Creando segmento de riesgo basado en Recencia y Gasto...

Distribución de segmentos de riesgo:


risk_segment
Mixto           7547
Desconectado    6202
Regular          987
Leal             264
Name: count, dtype: int64


Tasa de churn por segmento de riesgo:


risk_segment
Desconectado    0.219284
Mixto           0.109580
Leal            0.090909
Regular         0.088146
Name: churn, dtype: float64

## Interpretación de Segmentación de riesgo

#### **Distribución:**
- Mixto (50.3%), Desconectado (41.3%), Regular (6.6%), Leal (1.8%).

#### Tasa de churn por segmento:
- Desconectado: 21.9% (el más alto).
- Mixto: 11.0%.
- Leal: 9.1%.
- Regular: 8.8%.

### **Conclusión:** 
- La variable risk_segment captura perfectamente el riesgo de abandono. 
- Los clientes "Desconectados" (recencia > 90 días y gasto bajo) tienen más del doble de probabilidad de churn que los "Leales". 
- Esta variable debe incluirse como predictora categórica en el modelo.

---


## 📘 DICCIONARIO DE DATOS COMPLETO DESPUES DE LOS BLOQUES 1-5 Feature Engineering & Diagnóstico Avanzado

A continuación se presenta el diccionario de datos que abarca **todas las variables que existieron en el proyecto**, desde su estado original hasta su estado final del bloque 1 al 5.

| ID | Nombre Original | Tipo | Descripción | Estado | Notas / Transformación |
|:---|:---|:---|:---|:---|:---|
| 1 | `customer_id` | int64 | Identificador único del cliente. | **Eliminada** | Eliminada por ser un identificador sin valor predictivo. |
| 2 | `gender` | object | Género del cliente. | **Mantenida** | Se mantiene como categórica. Se imputarán nulos con la moda. |
| 3 | `age` | float64 | Edad del cliente. | **Eliminada** | Eliminada por alta multicolinealidad (VIF > 10) con `tenure_days`. |
| 4 | `country` | object | País de residencia. | **Eliminada** | No significativa en Chi-cuadrado (p=0.192). |
| 5 | `city` | object | Ciudad de residencia. | **Eliminada** | No significativa en Chi-cuadrado (p=0.677). |
| 6 | `signup_date` | datetime | Fecha de registro. | **Eliminada** | Se usó para crear `tenure_days`; ya no es necesaria. |
| 7 | `last_purchase_date` | datetime | Fecha de última compra. | **Eliminada** | Se usó para crear `recency_days`; ya no es necesaria. |
| 8 | `acquisition_channel` | object | Canal de adquisición. | **Eliminada** | No significativa en Chi-cuadrado (p=0.711). |
| 9 | `device_type` | object | Tipo de dispositivo. | **Eliminada** | No significativa en Chi-cuadrado (p=0.425). |
| 10 | `subscription_type` | object | Tipo de suscripción. | **Eliminada** | No significativa en Chi-cuadrado (p=0.466). |
| 11 | `is_premium_user` | int64 | Usuario premium. | **Mantenida** | Bajo VIF y sentido de negocio. |
| 12 | `total_visits` | int64 | Número total de visitas. | **Eliminada** | Alta multicolinealidad (VIF > 10) con `avg_session_time`. |
| 13 | `avg_session_time` | float64 | Tiempo promedio de sesión. | **Mantenida** | Captura engagement. |
| 14 | `pages_per_session` | float64 | Páginas por sesión. | **Mantenida** | Similar a anterior. |
| 15 | `email_open_rate` | float64 | Tasa de apertura de emails. | **Mantenida** | VIF bajo. Mide interacción. |
| 16 | `email_click_rate` | float64 | Tasa de clics en emails. | **Mantenida** | VIF bajo. Mide interacción. |
| 17 | `total_spent` | float64 | Gasto total acumulado. | **Mantenida** | Alta correlación con churn (-0.26). |
| 18 | `avg_order_value` | float64 | Valor promedio por pedido. | **Mantenida** | Complementa a `total_spent`. |
| 19 | `discount_used` | int64 | Usó descuento. | **Mantenida** | VIF bajo. |
| 20 | `coupon_code` | object | Código de cupón. | **Mantenida** | Se imputarán nulos con "None". |
| 21 | `support_tickets` | int64 | Número de tickets de soporte. | **Mantenida** | Correlación positiva con churn (+0.086). |
| 22 | `refund_requested` | int64 | Solicitó reembolso. | **Mantenida** | VIF bajo. |
| 23 | `delivery_delay_days` | int64 | Días de retraso en entrega. | **Mantenida** | VIF moderado. |
| 24 | `payment_method` | object | Método de pago. | **Eliminada** | No significativa en Chi-cuadrado (p=0.585). |
| 25 | `satisfaction_score` | float64 | Puntuación de satisfacción. | **Mantenida** | Correlación más alta (-0.262). |
| 26 | `nps_score` | int64 | Net Promoter Score. | **Mantenida** | VIF bajo. |
| 27 | `marketing_spend_per_user` | float64 | Gasto de marketing por usuario. | **Mantenida** | VIF moderado (6.5). |
| 28 | `lifetime_value` | float64 | Valor de vida del cliente. | **Mantenida** | VIF bajo (4.3). |
| 29 | `last_3_month_purchase_freq` | int64 | Frecuencia de compras recientes. | **Mantenida** | VIF bajo. |
| 30 | `churn` | int64 | Variable objetivo. | **Mantenida** | Se usa como target, no como predictor. |
| 31 | `recency_days` | int64 | **Nueva** – Días desde última compra. | **Creada (Bloque 1)** | Creada a partir de `last_purchase_date` en el notebook 02, Bloque 1. Captura abandono silencioso. |
| 32 | `tenure_days` | int64 | **Nueva** – Días desde el registro. | **Creada (Bloque 1)** | Creada a partir de `signup_date` en el notebook 02, Bloque 1. Captura lealtad. |
| 33 | `avg_spend_per_visit` | float64 | **Nueva** – Gasto por visita. | **Creada (Bloque 4)** | Creada en el notebook 02, Bloque 4. Mide eficiencia de conversión. |
| 34 | `ticket_rate` | float64 | **Nueva** – Tickets por visita. | **Creada (Bloque 4)** | Creada en el notebook 02, Bloque 4. Mide frustración relativa. |
| 35 | `recency_tenure_ratio` | float64 | **Nueva** – Proporción de vida sin comprar. | **Creada (Bloque 4)** | Creada en el notebook 02, Bloque 4. Indica qué porcentaje de la vida del cliente lleva inactivo. |
| 36 | `spend_per_ticket` | float64 | **Nueva** – Gasto por ticket de soporte. | **Creada (Bloque 4)** | Creada en el notebook 02, Bloque 4. Relaciona el gasto con la frustración. |
| 37 | `risk_segment` | object | **Nueva** – Segmento de riesgo. | **Creada (Bloque 5)** | Creada en el notebook 02, Bloque 5, mediante reglas de negocio (recencia + gasto). Categorías: Leal, Regular, Desconectado, Mixto. |

---



# Bloque 6: Guardado del Dataset Enriquecido

- Guardamos el dataset con las nuevas variables para que estén disponibles para el pipeline de modelado. 

No se imputan valores nulos aquí; la imputación se hará dentro del ColumnTransformer para evitar fuga de datos.

In [8]:
# =====================================================
# Bloque 6: Guardado del Dataset Enriquecido (CORREGIDO)
# =====================================================
print("\n💾 Guardando dataset enriquecido...")

# Actualizar lista de columnas numéricas y categóricas con las nuevas variables
new_numeric = ['recency_days', 'tenure_days', 'avg_spend_per_visit', 'ticket_rate', 'recency_tenure_ratio', 'spend_per_ticket']
new_categorical = ['risk_segment']

# Guardar en processed con NOMBRE DESCRIPTIVO
ENRICHED_DATA_PATH = "../data/processed/data_enriched.csv"
eda.to_csv(ENRICHED_DATA_PATH, index=False)   # <--- CAMBIADO AQUÍ
print(f"✅ Dataset enriquecido guardado en {ENRICHED_DATA_PATH}")
print(f"Dimensiones: {eda.shape}")
print("📌 Nota: Los valores nulos se mantienen. La imputación se realizará en el pipeline de modelado.")


💾 Guardando dataset enriquecido...
✅ Dataset enriquecido guardado en ../data/processed/data_enriched.csv
Dimensiones: (15000, 37)
📌 Nota: Los valores nulos se mantienen. La imputación se realizará en el pipeline de modelado.


# PIPELINE DE PREPROCESAMIENTO

- Seleccionar las variables definitivas (excluyendo las redundantes).
- Definir transformadores para numéricas y categóricas.
- Construir un `ColumnTransformer` con imputación, escalado y codificación.
- Dividir el dataset en entrenamiento y prueba (estratificado).
- Guardar los splits y el preprocesador para usarlos en `03_Modeling.`

# Bloque 7 — Selección de variables definitivas

- Basado en el análisis de VIF y Chi-cuadrado, eliminamos variables redundantes o no significativas. 

Mantenemos las que tienen sentido de negocio y las nuevas creadas.

In [9]:
# =====================================================
# Bloque 7: Selección de variables definitivas
# =====================================================
print("\n🎯 Seleccionando variables para el modelo...")

# Variables a eliminar (redundantes o no significativas)
cols_to_drop = [
    'customer_id',          # Identificador
    'signup_date',          # Ya no necesaria (tenemos tenure)
    'last_purchase_date',   # Ya no necesaria (tenemos recency)
    'age',                  # VIF alto y redundante con tenure
    'total_visits',         # VIF alto y redundante con avg_session_time
    'city',                 # No significativa en Chi-cuadrado
    'country',              # No significativa
    'subscription_type',    # No significativa
    'device_type',          # No significativa
    'acquisition_channel',  # No significativa
    'payment_method',       # No significativa
    'churn'
]

# Crear DataFrame con variables seleccionadas
X = eda.drop(columns=cols_to_drop, errors='ignore')
y = eda['churn'].copy()

print(f"Variables predictoras finales: {X.shape[1]}")
print(X.columns.tolist())


🎯 Seleccionando variables para el modelo...
Variables predictoras finales: 25
['gender', 'is_premium_user', 'avg_session_time', 'pages_per_session', 'email_open_rate', 'email_click_rate', 'total_spent', 'avg_order_value', 'discount_used', 'coupon_code', 'support_tickets', 'refund_requested', 'delivery_delay_days', 'satisfaction_score', 'nps_score', 'marketing_spend_per_user', 'lifetime_value', 'last_3_month_purchase_freq', 'recency_days', 'tenure_days', 'avg_spend_per_visit', 'ticket_rate', 'recency_tenure_ratio', 'spend_per_ticket', 'risk_segment']


# Selección de Variables Definitivas

## Lista de variables finales (25)

| # | Variable | Tipo | Origen | Justificación de permanencia |
|---|----------|------|--------|-------------------------------|
| 1 | `gender` | Categórica | Original | Única categórica significativa (Chi‑cuadrado p=0.017). |
| 2 | `is_premium_user` | Numérica | Original | Baja multicolinealidad (VIF≈1.9) y sentido de negocio. |
| 3 | `avg_session_time` | Numérica | Original | VIF moderado (7.6). Captura engagement. |
| 4 | `pages_per_session` | Numérica | Original | VIF moderado (7.7). Similar a lo anterior. |
| 5 | `email_open_rate` | Numérica | Original | VIF bajo. Mide interacción digital. |
| 6 | `email_click_rate` | Numérica | Original | VIF bajo. Mide interacción digital. |
| 7 | `total_spent` | Numérica | Original | Correlación alta con Churn (-0.26). Se mantiene. |
| 8 | `avg_order_value` | Numérica | Original | VIF moderado (6.4). Complementa a `total_spent`. |
| 9 | `discount_used` | Numérica | Original | VIF bajo (1.9). Indicador de sensibilidad al precio. |
| 10 | `coupon_code` | Categórica | Original | Aunque tiene 41% de nulos, se convierte en "None". Captura comportamiento de compra. |
| 11 | `support_tickets` | Numérica | Original | Correlación positiva (+0.086). Indicador de frustración. |
| 12 | `refund_requested` | Numérica | Original | VIF bajo. Indicador de insatisfacción. |
| 13 | `delivery_delay_days` | Numérica | Original | VIF moderado. Impacto en la experiencia. |
| 14 | `satisfaction_score` | Numérica | Original | Correlación más alta (-0.262). Predictor estrella. |
| 15 | `nps_score` | Numérica | Original | VIF bajo. Complementa a satisfacción. |
| 16 | `marketing_spend_per_user` | Numérica | Original | VIF moderado (6.5). Inversión en marketing. |
| 17 | `lifetime_value` | Numérica | Original | VIF bajo (4.3). Se mantiene porque no alcanzó el umbral de 10. |
| 18 | `last_3_month_purchase_freq` | Numérica | Original | VIF bajo. Frecuencia reciente. |
| 19 | `recency_days` | Numérica | Nueva | Captura el abandono silencioso. Predictor clave en retail. |
| 20 | `tenure_days` | Numérica | Nueva | Captura la lealtad acumulada. |
| 21 | `avg_spend_per_visit` | Numérica | Nueva (Ing.) | Eficiencia de conversión. |
| 22 | `ticket_rate` | Numérica | Nueva (Ing.) | Frustración relativa por visita. |
| 23 | `recency_tenure_ratio` | Numérica | Nueva (Ing.) | Proporción de vida sin comprar. |
| 24 | `spend_per_ticket` | Numérica | Nueva (Ing.) | Gasto por ticket de soporte. |
| 25 | `risk_segment` | Categórica | Nueva (Ing.) | Segmento de riesgo (Leal, Regular, Desconectado, Mixto). |

---

## Variables eliminadas (11) y justificación

| Variable | Tipo | Razón de eliminación |
|----------|------|-----------------------|
| `customer_id` | Identificador | No es un predictor, solo identifica al cliente. |
| `signup_date` | Fecha | Redundante con `tenure_days` (ya creada a partir de esta fecha). |
| `last_purchase_date` | Fecha | Redundante con `recency_days` (ya creada a partir de esta fecha). |
| `age` | Numérica | Alta multicolinealidad (VIF = 11.47). Correlacionada con `tenure_days` y otras variables. |
| `total_visits` | Numérica | Alta multicolinealidad (VIF = 13.45). Su información está contenida en `avg_session_time` y `pages_per_session`. |
| `city` | Categórica | No significativa en Chi‑cuadrado (p=0.677). No hay evidencia de relación con churn. |
| `country` | Categórica | No significativa en Chi‑cuadrado (p=0.192). No hay evidencia de relación con churn. |
| `subscription_type` | Categórica | No significativa en Chi‑cuadrado (p=0.466). |
| `device_type` | Categórica | No significativa en Chi‑cuadrado (p=0.425). |
| `acquisition_channel` | Categórica | No significativa en Chi‑cuadrado (p=0.711). |
| `payment_method` | Categórica | No significativa en Chi‑cuadrado (p=0.585). |

---

## Impacto de la eliminación

Este subconjunto de **25 variables** constituye el conjunto óptimo de predictores. Al eliminar las variables redundantes o no informativas, se logra:

- **Menor dimensionalidad**: menos ruido, menor riesgo de sobreajuste.
- **Entrenamiento más rápido**: menos columnas → menor tiempo de cómputo en modelos como XGBoost o Random Forest.
- **Mayor interpretabilidad**: cada variable que se mantiene tiene una justificación clara, ya sea por su poder predictivo (`satisfaction_score`, `total_spent`) o por su relevancia de negocio (`risk_segment`, `recency_days`).

---

# Implicaciones de la Selección de Variables para el Modelo

## 7.1. Agrupación de variables finales

| Grupo | Variables incluidas | Justificación | Rol en el modelo |
|-------|---------------------|---------------|------------------|
| **Comportamiento base** | `avg_session_time`, `pages_per_session`, `email_open_rate`, `email_click_rate`, `total_spent`, `avg_order_value`, `discount_used`, `support_tickets`, `refund_requested`, `delivery_delay_days`, `satisfaction_score`, `nps_score`, `marketing_spend_per_user`, `lifetime_value`, `last_3_month_purchase_freq` | Capturan el engagement, gasto, satisfacción y calidad del servicio. Son el núcleo del modelo. | Aportan la señal principal para discriminar entre churners y no churners. Son la base de la predicción. |
| **Temporales (nuevas)** | `recency_days`, `tenure_days` | Miden el abandono silencioso (recencia) y la lealtad acumulada (antigüedad). | Incorporan la dimensión temporal del comportamiento, clave para detectar patrones de abandono. |
| **Interacciones (nuevas)** | `avg_spend_per_visit`, `ticket_rate`, `recency_tenure_ratio`, `spend_per_ticket` | Capturan eficiencia, frustración relativa y proporción de vida inactiva. | Mejoran la separabilidad al combinar información de varias variables originales en indicadores compuestos. |
| **Categóricas** | `gender`, `coupon_code`, `risk_segment` | Solo `gender` fue significativa; `coupon_code` y `risk_segment` se mantienen por lógica de negocio. | Añaden información cualitativa que los modelos basados en árboles pueden aprovechar (interacciones con otras variables). |


## 7.2. ¿Qué significa esto para el modelo?

### 7.2.1. Dimensionalidad reducida y menor ruido
- **Antes**: 36 variables originales (incluyendo irrelevantes y redundantes).  
- **Ahora**: 25 variables, con eliminación de 11 variables por alta multicolinealidad, falta de significancia estadística o redundancia.  
- **Impacto**: Menor riesgo de sobreajuste, ya que se eliminan variables que aportan ruido o información duplicada.

### 7.2.2. Entrenamiento más rápido y eficiente
- Menos columnas → menor consumo de memoria y tiempo de cómputo, especialmente en algoritmos como XGBoost o Random Forest que realizan búsquedas de splits.  
- La reducción es modesta (25 vs 36), pero suficiente para acelerar la validación cruzada y la búsqueda de hiperparámetros.

### 7.2.3. Mayor interpretabilidad
- Cada variable que permanece tiene una justificación clara (estadística o de negocio).  
- Las nuevas variables de ingeniería (`recency_tenure_ratio`, `ticket_rate`, etc.) tienen un significado directo para el negocio, lo que facilita la explicación del modelo a stakeholders.

### 7.2.4. Compatibilidad con modelos no lineales
- Dado que las correlaciones con `churn` son bajas (salvo `satisfaction_score` y `total_spent`), el conjunto de variables está diseñado para ser explotado por **modelos no lineales** (árboles, boosting, SVM con kernel).  
- Las interacciones creadas manualmente (ej. `recency_tenure_ratio`) ayudan a capturar relaciones complejas que los modelos lineales no podrían detectar.

### 7.2.5. Manejo de categóricas
- Solo `gender` resultó significativa; `coupon_code` y `risk_segment` se mantienen por criterio de negocio.  
- Esto implica que el modelo no dependerá de muchas variables categóricas, lo que reduce la dimensionalidad tras la codificación one‑hot.



## 7.3. Riesgos y consideraciones

- **Posible pérdida de información**: Aunque `age` y `total_visits` se eliminaron por VIF alto, su información está parcialmente contenida en `tenure_days`, `avg_session_time` y `pages_per_session`. No se espera una merma significativa en el rendimiento.  
- **Sesgo de selección**: La selección se basó en VIF y Chi‑cuadrado sobre el conjunto completo; se deberá verificar que estas variables mantengan su poder predictivo en la partición de test (validación cruzada).  
- **Ingeniería de características**: Las nuevas variables deben ser recalculadas exactamente igual en producción; su estabilidad debe ser monitoreada.


## 7.4. Conclusión para el modelado

El subconjunto de 25 variables está **óptimamente depurado** y listo para ser introducido en el pipeline de modelado. Se espera que:

- Modelos basados en árboles (Random Forest, XGBoost, LightGBM) obtengan un buen rendimiento sin necesidad de escalado adicional (aunque se aplicará `RobustScaler` por robustez).
- La importancia de características revelará que `satisfaction_score`, `total_spent`, `recency_days` y `ticket_rate` estarán entre los predictores más influyentes.
- El tiempo de entrenamiento será aceptable y la interpretabilidad será alta, facilitando la comunicación de resultados.

El siguiente paso es proceder al **notebook 03_Modeling** con estos datos preprocesados.

---

# Bloque 8 — Definición de tipos y pipelines

- Separamos las columnas en numéricas y categóricas según su tipo. 
- Para las numéricas aplicaremos imputación con mediana y escalado estándar. 
- Para las categóricas, imputación con moda y codificación one-hot.

In [10]:
# =====================================================
# Bloque 8: Definición de pipelines por tipo de variable
# =====================================================
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("\n⚙️ Definiendo pipelines de transformación...")

# Identificar columnas numéricas y categóricas (después de eliminar)
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Eliminar 'churn' de las listas (ya está en y)
if 'churn' in num_cols:
    num_cols.remove('churn')
if 'churn' in cat_cols:
    cat_cols.remove('churn')

print(f"Numéricas: {len(num_cols)}")
print(f"Categóricas: {len(cat_cols)}")

# Pipeline para variables numéricas
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para variables categóricas
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer: aplica cada pipeline a su grupo de columnas
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

print("✅ Preprocesador definido.")


⚙️ Definiendo pipelines de transformación...
Numéricas: 22
Categóricas: 3
✅ Preprocesador definido.


# Definición de pipelines por tipo de variable

## ¿Qué se hizo?

Se crearon dos pipelines:

- **Pipeline numérico** (22 variables): imputación con mediana → escalado con `StandardScaler`.
- **Pipeline categórico** (3 variables): imputación con moda → codificación one‑hot.


El `ColumnTransformer` combina ambos, produciendo **32 columnas** después de la transformación (22 numéricas escaladas + 10 columnas one‑hot).



## Efecto de cada transformación en el modelo

| Transformación | Variable(s) afectadas | Efecto en el modelo |
|----------------|------------------------|----------------------|
| **Imputación con mediana** | Numéricas con nulos (ej. `total_spent`, `age` corregida) | Evita que el modelo falle por `NaN`. La mediana es robusta ante outliers. |
| **Imputación con moda** | `gender` | Asigna el valor más frecuente a los nulos. Es la opción menos riesgosa. |
| **Imputación con "None"** | `coupon_code` (se hará en bloque previo, no aquí) | Convierte los faltantes en una categoría válida ("sin cupón"). |
| **Escalado (`StandardScaler`)** | Todas las numéricas | Centraliza en media 0 y desviación 1. Es obligatorio para modelos basados en distancias (SVM, k‑NN, Regresión Logística). |
| **One‑Hot Encoding** | `gender` (3), `coupon_code` (4), `risk_segment` (4) | Convierte categorías en columnas binarias (0/1). Evita que el modelo asuma un orden inexistente. |



## Impacto

El preprocesador garantiza que los datos de entrada estén en el formato óptimo para cualquier modelo de ML. Además, al estar encapsulado en un `Pipeline`, se aplicará de forma idéntica a los datos de prueba sin filtrar información.

---

# Bloque 9 — División train/test y aplicación del preprocesador

- Dividimos los datos en entrenamiento (80%) y prueba (20%) de forma estratificada para preservar la proporción de churn. Luego, ajustamos el preprocesador solo en el conjunto de entrenamiento y transformamos ambos conjuntos.

In [11]:
# =====================================================
# Bloque 9: División train/test y preprocesamiento
# =====================================================
from sklearn.model_selection import train_test_split

print("\n📊 Dividiendo datos en entrenamiento y prueba...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Entrenamiento: {X_train.shape[0]} registros")
print(f"Prueba: {X_test.shape[0]} registros")

# Ajustar el preprocesador en entrenamiento y transformar ambos
print("\n🔄 Aplicando preprocesamiento...")
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print(f"Shape después de preprocesar: {X_train_prep.shape}")


📊 Dividiendo datos en entrenamiento y prueba...
Entrenamiento: 12000 registros
Prueba: 3000 registros

🔄 Aplicando preprocesamiento...
Shape después de preprocesar: (12000, 32)



# División train/test y preprocesamiento

## ¿Qué se hizo?

Se dividió el dataset en **80% entrenamiento** (12,000 registros) y **20% prueba** (3,000 registros) de forma estratificada (respetando la proporción de `churn`). Luego, se aplicó el `ColumnTransformer` (`fit_transform` en entrenamiento, `transform` en prueba).

**Resultado:** `X_train_prep` (12,000 × 32) y `X_test_prep` (3,000 × 32).


## Implicaciones para el modelo

| Aspecto | Implicación |
|---------|-------------|
| **Tamaño de entrenamiento (12k)** | Suficiente para entrenar modelos complejos (Random Forest, XGBoost) sin sobreajuste severo. |
| **Tamaño de prueba (3k)** | Permite una evaluación robusta del rendimiento (métrica estable). |
| **Estratificación** | Asegura que la clase minoritaria (Churn) esté representada en la misma proporción (~15.3%) en ambos conjuntos. Sin esto, el test podría tener 0 churners y la evaluación sería inútil. |
| **Preprocesamiento en entrenamiento** | El escalador aprende la media/desviación de los datos de entrenamiento y la aplica a prueba. Esto evita la fuga de datos (no se usa información de prueba para escalar). |


## Impacto

El modelo se entrena con datos representativos y se evalúa de forma justa y sin sesgos.

---

## Conocer las 32 variables finales.

In [12]:
# Obtener nombres de las columnas one-hot generadas
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']
cat_feature_names = cat_encoder.get_feature_names_out(cat_cols)

# Lista completa de nombres (numéricas + one-hot)
final_feature_names = num_cols + list(cat_feature_names)

print("Número total de columnas:", len(final_feature_names))
print("Nombres de las columnas:")
print(final_feature_names)

Número total de columnas: 32
Nombres de las columnas:
['is_premium_user', 'avg_session_time', 'pages_per_session', 'email_open_rate', 'email_click_rate', 'total_spent', 'avg_order_value', 'discount_used', 'support_tickets', 'refund_requested', 'delivery_delay_days', 'satisfaction_score', 'nps_score', 'marketing_spend_per_user', 'lifetime_value', 'last_3_month_purchase_freq', 'recency_days', 'tenure_days', 'avg_spend_per_visit', 'ticket_rate', 'recency_tenure_ratio', 'spend_per_ticket', 'gender_Female', 'gender_Male', 'gender_Other', 'coupon_code_NEW20', 'coupon_code_REF10', 'coupon_code_SALE15', 'risk_segment_Desconectado', 'risk_segment_Leal', 'risk_segment_Mixto', 'risk_segment_Regular']


## 📘 DICCIONARIO DE VARIABLES FINALES — 32 COLUMNAS

Basado en la lista proporcionada, este diccionario documenta cada una de las 32 columnas que ingresan al modelo después del preprocesamiento (imputación, escalado y codificación one-hot). Incluye identificación, origen, transformación aplicada, tipo y una breve descripción funcional.


### 🔹 Diccionario completo

| ID | Nombre en el modelo | Variable original | Transformación | Tipo | Descripción / Origen |
|:---|:---|:---|:---|:---|:---|
| 1 | `is_premium_user` | `is_premium_user` | Escalada (StandardScaler) | Numérica | Indicador de usuario premium (1 = Sí, 0 = No). Original. |
| 2 | `avg_session_time` | `avg_session_time` | Escalada (StandardScaler) | Numérica | Tiempo promedio de sesión en minutos. Original. |
| 3 | `pages_per_session` | `pages_per_session` | Escalada (StandardScaler) | Numérica | Número promedio de páginas por sesión. Original. |
| 4 | `email_open_rate` | `email_open_rate` | Escalada (StandardScaler) | Numérica | Tasa de apertura de correos (0-1). Original. |
| 5 | `email_click_rate` | `email_click_rate` | Escalada (StandardScaler) | Numérica | Tasa de clics en correos (0-1). Original. |
| 6 | `total_spent` | `total_spent` | Escalada (StandardScaler) | Numérica | Gasto total acumulado del cliente. Original. |
| 7 | `avg_order_value` | `avg_order_value` | Escalada (StandardScaler) | Numérica | Valor promedio por pedido. Original. |
| 8 | `discount_used` | `discount_used` | Escalada (StandardScaler) | Numérica | Indicador de uso de descuento (1 = Sí). Original. |
| 9 | `support_tickets` | `support_tickets` | Escalada (StandardScaler) | Numérica | Número de tickets de soporte abiertos. Original. |
| 10 | `refund_requested` | `refund_requested` | Escalada (StandardScaler) | Numérica | Indicador de solicitud de reembolso (1 = Sí). Original. |
| 11 | `delivery_delay_days` | `delivery_delay_days` | Escalada (StandardScaler) | Numérica | Días de retraso en la entrega. Original. |
| 12 | `satisfaction_score` | `satisfaction_score` | Escalada (StandardScaler) | Numérica | Puntuación de satisfacción (1-5). Original. |
| 13 | `nps_score` | `nps_score` | Escalada (StandardScaler) | Numérica | Net Promoter Score (0-10). Original. |
| 14 | `marketing_spend_per_user` | `marketing_spend_per_user` | Escalada (StandardScaler) | Numérica | Gasto de marketing atribuido por usuario. Original. |
| 15 | `lifetime_value` | `lifetime_value` | Escalada (StandardScaler) | Numérica | Valor de vida estimado del cliente (LTV). Original. |
| 16 | `last_3_month_purchase_freq` | `last_3_month_purchase_freq` | Escalada (StandardScaler) | Numérica | Frecuencia de compras en los últimos 3 meses. Original. |
| 17 | `recency_days` | `recency_days` | Escalada (StandardScaler) | Numérica | Días desde la última compra. **Creada** en Bloque 1 (feature engineering). |
| 18 | `tenure_days` | `tenure_days` | Escalada (StandardScaler) | Numérica | Días desde el registro (antigüedad). **Creada** en Bloque 1 (feature engineering). |
| 19 | `avg_spend_per_visit` | `avg_spend_per_visit` | Escalada (StandardScaler) | Numérica | Gasto promedio por visita. **Creada** en Bloque 4 (interacción). |
| 20 | `ticket_rate` | `ticket_rate` | Escalada (StandardScaler) | Numérica | Tickets de soporte por visita. **Creada** en Bloque 4 (interacción). |
| 21 | `recency_tenure_ratio` | `recency_tenure_ratio` | Escalada (StandardScaler) | Numérica | Proporción de vida del cliente sin comprar (recency/tenure). **Creada** en Bloque 4. |
| 22 | `spend_per_ticket` | `spend_per_ticket` | Escalada (StandardScaler) | Numérica | Gasto por ticket de soporte (total_spent / (tickets+1)). **Creada** en Bloque 4. |
| 23 | `gender_Female` | `gender` | One-Hot Encoding | Binaria | Género: Female (1 = sí). Categoría base omitida (Male es referencia implícita). |
| 24 | `gender_Male` | `gender` | One-Hot Encoding | Binaria | Género: Male (1 = sí). |
| 25 | `gender_Other` | `gender` | One-Hot Encoding | Binaria | Género: Other (1 = sí). |
| 26 | `coupon_code_NEW20` | `coupon_code` | One-Hot Encoding | Binaria | Código de cupón: NEW20. |
| 27 | `coupon_code_REF10` | `coupon_code` | One-Hot Encoding | Binaria | Código de cupón: REF10. |
| 28 | `coupon_code_SALE15` | `coupon_code` | One-Hot Encoding | Binaria | Código de cupón: SALE15. |
| 29 | `risk_segment_Desconectado` | `risk_segment` | One-Hot Encoding | Binaria | Segmento de riesgo: Desconectado (recencia alta, gasto bajo). |
| 30 | `risk_segment_Leal` | `risk_segment` | One-Hot Encoding | Binaria | Segmento de riesgo: Leal (recencia baja, gasto alto). |
| 31 | `risk_segment_Mixto` | `risk_segment` | One-Hot Encoding | Binaria | Segmento de riesgo: Mixto (casos intermedios). |
| 32 | `risk_segment_Regular` | `risk_segment` | One-Hot Encoding | Binaria | Segmento de riesgo: Regular (riesgo medio). |



### 📌 Notas explicativas

- **Columnas numéricas (1–22):** Todas fueron **escaladas con `StandardScaler`** (media 0, desviación estándar 1) para que el modelo no se vea afectado por las diferentes escalas originales.
- **Columnas binarias (23–32):** Corresponden a variables categóricas transformadas mediante **One-Hot Encoding**. Se omitió una categoría de referencia en cada caso para evitar multicolinealidad perfecta:
  - `gender` → referencia implícita: no se creó columna para una categoría (probablemente la que no apareció en entrenamiento o la primera alfabéticamente).
  - `coupon_code` → se generaron 3 de las 4 categorías originales; la categoría faltante (quizás `None` o `REF20`) actúa como referencia.
  - `risk_segment` → se generaron 4 categorías; la referencia implícita es la categoría no listada o la que no apareció en entrenamiento.

- **Total de columnas:** 22 numéricas + 10 binarias = **32 columnas** finales.



### ✅ ¿Cómo se generó este diccionario?

Este diccionario se creó a partir de la lista de nombres proporcionada, combinada con la información de los bloques anteriores (variables originales, transformaciones y origen de las variables nuevas). Corresponde exactamente a las columnas que recibe el modelo en `X_train_prep` y `X_test_prep`.

---



# Bloque 10 — Guardado de artefactos

Guardamos los datos preprocesados y el preprocesador ajustado para poder reutilizarlos en el notebook de modelado sin tener que repetir el ajuste.

In [13]:
# =====================================================
# Bloque 10: Guardado de artefactos
# =====================================================
import joblib
import os

print("\n💾 Guardando artefactos...")

# Crear carpeta models/ si no existe
os.makedirs("../models", exist_ok=True)

# Guardar preprocesador ajustado
joblib.dump(preprocessor, "../models/preprocessor.joblib")

# Guardar splits preprocesados
np.save("../data/processed/X_train_prep.npy", X_train_prep)
np.save("../data/processed/X_test_prep.npy", X_test_prep)
np.save("../data/processed/y_train.npy", y_train.values)
np.save("../data/processed/y_test.npy", y_test.values)

# También guardar los nombres de las columnas (opcional, para interpretación)
feature_names = num_cols + list(preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(cat_cols))
with open("../data/processed/feature_names.txt", "w") as f:
    f.write("\n".join(feature_names))

print("✅ Artefactos guardados:")
print("   - preprocessor.joblib")
print("   - X_train_prep.npy, X_test_prep.npy")
print("   - y_train.npy, y_test.npy")
print("   - feature_names.txt")


💾 Guardando artefactos...
✅ Artefactos guardados:
   - preprocessor.joblib
   - X_train_prep.npy, X_test_prep.npy
   - y_train.npy, y_test.npy
   - feature_names.txt


# Guardado de artefactos

## ¿Qué función tiene este bloque?

**Persistencia.** Convierte los objetos procesados en archivos físicos en disco, para que el notebook `03_Modeling` pueda cargarlos y continuar sin tener que re‑ejecutar todo el preprocesamiento.


## Archivos guardados: nombres y ubicación

| Archivo | Ruta | Contenido |
|---------|------|-----------|
| `preprocessor.joblib` | `models/` | El `ColumnTransformer` ya ajustado (con las medianas, escalas y categorías aprendidas). |
| `X_train_prep.npy` | `data/processed/` | Matriz NumPy de entrenamiento (12,000 × 32). |
| `X_test_prep.npy` | `data/processed/` | Matriz NumPy de prueba (3,000 × 32). |
| `y_train.npy` | `data/processed/` | Vector NumPy de etiquetas de entrenamiento. |
| `y_test.npy` | `data/processed/` | Vector NumPy de etiquetas de prueba. |
| `feature_names.txt` | `data/processed/` | Lista de nombres de las 32 columnas (para interpretación). |


## ¿De dónde carga `03_Modeling`?

El notebook `03_Modeling` cargará los datos desde `data/processed/` (los archivos `.npy`) y el preprocesador desde `models/preprocessor.joblib`.

---

# Bloque 11 — Verificación rápida

Comprobamos que los datos preprocesados no tengan nulos y que las escalas sean razonables.

In [14]:
# =====================================================
# Bloque 11: Verificación rápida
# =====================================================
print("\n🔍 Verificando datos preprocesados...")

# Verificar nulos en los datos transformados
print(f"Nulos en X_train_prep: {np.isnan(X_train_prep).sum()}")
print(f"Nulos en X_test_prep: {np.isnan(X_test_prep).sum()}")

# Mostrar estadísticas básicas de las primeras variables
print("\nEstadísticas de X_train_prep (primeras 5 columnas):")
print(pd.DataFrame(X_train_prep[:, :5]).describe().round(2))


🔍 Verificando datos preprocesados...
Nulos en X_train_prep: 0
Nulos en X_test_prep: 0

Estadísticas de X_train_prep (primeras 5 columnas):
              0         1         2         3         4
count  12000.00  12000.00  12000.00  12000.00  12000.00
mean      -0.00     -0.00      0.00     -0.00      0.00
std        1.00      1.00      1.00      1.00      1.00
min       -0.67     -2.68     -2.70     -1.71     -1.74
25%       -0.67     -0.68     -0.68     -0.89     -0.84
50%       -0.67     -0.00     -0.00      0.01     -0.01
75%        1.49      0.68      0.68      0.87      0.89
max        1.49      3.71      4.60      1.73      1.72


# Verificación rápida

## ¿Qué se hizo?

Se verificaron dos aspectos sobre los datos preprocesados:

1. **Ausencia de valores nulos** en `X_train_prep` y `X_test_prep`.
2. **Estadísticas descriptivas** de las primeras 5 columnas (variables numéricas escaladas).

**Resultado:** 0 nulos. Media ≈ 0, desviación estándar ≈ 1 en las variables escaladas.


## Métricas de control de calidad

| Métrica | Valor | Implicación específica para el modelo |
|---------|-------|----------------------------------------|
| **Nulos** | 0 | La imputación funcionó correctamente. No hay `NaN` que provoquen errores en ningún modelo. |
| **Media** | ≈ 0 | `StandardScaler` centró los datos. Esto es necesario para modelos basados en distancias (SVM, k‑NN, regresión logística). |
| **Desviación estándar** | ≈ 1 | El escalado fue exitoso. Todas las variables numéricas están en la misma magnitud, lo que evita que variables con escalas mayores dominen el gradiente en modelos lineales. |
| **Rango (min/max)** | -2.7 a 4.6 | Los valores extremos (outliers) persisten, pero están dentro de un rango manejable. Para árboles y XGBoost es irrelevante (solo importa el orden). Para regresión logística/SVM, estos valores extremos pueden influir, pero menos que sin escalar. |


## Interpretación para el modelado

- **Modelos basados en árboles (Random Forest, XGBoost, LightGBM)**: El escalado no es necesario, pero no perjudica. La presencia de outliers no afecta el rendimiento porque los árboles dividen por umbrales. El preprocesamiento es compatible.
- **Modelos lineales (Regresión Logística, SVM lineal)**: El escalado es obligatorio. La media ≈0 y desviación ≈1 garantizan que los coeficientes sean comparables y que el descenso de gradiente converja correctamente. Los outliers en el rango [-2.7, 4.6] aún pueden influir, pero están atenuados.
- **Modelos basados en distancias (k‑NN, SVM con kernel RBF)**: El escalado es crítico. La escala uniforme evita que variables con mayor rango dominen la métrica de distancia. Los outliers pueden distorsionar la distancia, pero el escalado los reduce en magnitud.


## Conclusión

El preprocesamiento ha sido exitoso. Los datos están:

- **Completos** (sin nulos).
- **Escalados** (media 0, desv. 1).
- **Listos** para ser alimentados a cualquier modelo de clasificación supervisado.

El siguiente paso es proceder con el entrenamiento y evaluación en el notebook `03_Modeling`.
---
